In [2]:
# 提取对侧血管
import os
import re
import SimpleITK as sitk
from tqdm import tqdm

# --- 路径配置 ---
contralateral_root = "PROJECT_ROOT/对侧血管补充" 
output_reference = "PROJECT_ROOT/reference"

# --- 正则匹配逻辑 ---
# 匹配：姓名_ID（[远端/近端]参考层为[数字]）
# 捕获组：1=姓名_ID, 2=参考类型(远端/近端), 3=切片编号
pattern_contra = re.compile(r"(.+?)[（(](远端|近端)参考层为(\d+)[）)]")

os.makedirs(output_reference, exist_ok=True)

for patient_folder in tqdm(os.listdir(contralateral_root), desc="补充对侧参考层"):
    # 标准化字符
    fixed_name = (
        patient_folder.replace("　", " ")
        .replace("（", "(")
        .replace("）", ")")
        .strip()
    )

    match = pattern_contra.match(fixed_name)
    if not match:
        print(f"跳过格式不符文件夹: {patient_folder}")
        continue

    # 提取信息
    case_id_raw, ref_type_str, ref_slice_num = match.groups()
    ref_slice_idx = int(ref_slice_num)
    
    # 1. 确定文件名映射：远端 -> distal, 近端 -> proximal
    prefix = "distal" if ref_type_str == "远端" else "proximal"
    
    # 2. 统一 case_id 格式 (与主脚本一致)
    case_id = case_id_raw.strip().replace(' ', '_')
    
    case_path = os.path.join(contralateral_root, patient_folder)
    reference_case_dir = os.path.join(output_reference, case_id)
    os.makedirs(reference_case_dir, exist_ok=True)

    # --- 寻找文件 ---
    mask_file = None
    t1_file = None
    for f in os.listdir(case_path):
        if f.endswith("_contour_slice_prediction.mhd") and "T1CE" not in f:
            mask_file = os.path.join(case_path, f)
        elif f.endswith("_img.mhd") and "T1CE" not in f:
            t1_file = os.path.join(case_path, f)

    if not mask_file or not t1_file:
        print(f"文件缺失 {case_id}: Mask={bool(mask_file)}, T1={bool(t1_file)}")
        continue

    try:
        # --- 读取与切片提取 ---
        mask_img = sitk.ReadImage(mask_file)
        t1_img = sitk.ReadImage(t1_file)
        
        if ref_slice_idx < 1 or ref_slice_idx > mask_img.GetSize()[2]:
            print(f"索引越界 {case_id}: {ref_type_str}第 {ref_slice_idx} 层 (总层数 {mask_img.GetSize()[2]})")
            continue

        extractor = sitk.ExtractImageFilter()
        extractor.SetSize([mask_img.GetSize()[0], mask_img.GetSize()[1], 0])
        extractor.SetIndex([0, 0, ref_slice_idx - 1]) # 转为 0-based

        slice_mask = extractor.Execute(mask_img)
        slice_t1 = extractor.Execute(t1_img)

        # --- 分离管腔(1)和管壁(2) ---
        lumen_mask = sitk.BinaryThreshold(slice_mask, 1, 1, 1, 0)
        wall_mask = sitk.BinaryThreshold(slice_mask, 2, 2, 1, 0)

        # --- 保存：命名格式 [prefix]_lumen.mhd / [prefix]_wall.mhd ---
        sitk.WriteImage(lumen_mask, os.path.join(reference_case_dir, f"{prefix}_lumen.mhd"))
        sitk.WriteImage(wall_mask, os.path.join(reference_case_dir, f"{prefix}_wall.mhd"))
        sitk.WriteImage(slice_t1, os.path.join(reference_case_dir, f"{prefix}_t1.mhd"))

    except Exception as e:
        print(f"处理 {case_id} 时出错: {e}")

print("对侧参考层（远端/近端）补充提取完成")

补充对侧参考层: 100%|██████████| 13/13 [00:01<00:00,  8.45it/s]

对侧参考层（远端/近端）补充提取完成


In [6]:
import os
import re
import numpy as np
import SimpleITK as sitk
import pandas as pd
from sklearn.linear_model import LinearRegression

# ========== 1. 基础计算工具 ==========
def get_2d_area_fixed(file_path):
    """根据固定间距 (0.1, 0.1) 计算面积"""
    if not os.path.exists(file_path) or file_path is None:
        return None
    try:
        img = sitk.ReadImage(file_path)
        data = sitk.GetArrayFromImage(img)
        # 面积 = 像素数 * (0.1 * 0.1)
        return np.sum(data > 0) * 0.01
    except:
        return None

def get_z_idx(filename):
    """提取文件名中的 slice_XX 数字"""
    match = re.search(r'slice_(\d+)', filename)
    return int(match.group(1)) if match else None

# ========== 2. 寻找全段最薄管壁层面 ==========
def find_thinnest_wall_site(wall_folder):
    """
    遍历该病例所有管壁切片，找到面积最小的切片作为参考层
    """
    files = [f for f in os.listdir(wall_folder) if f.endswith(".mhd")]
    min_wa = float('inf')
    ref_file = None
    ref_z = None

    for f in files:
        wa = get_2d_area_fixed(os.path.join(wall_folder, f))
        z = get_z_idx(f)
        if wa is not None and wa < min_wa and wa > 0:
            min_wa = wa
            ref_file = f
            ref_z = z
            
    return ref_file, min_wa, ref_z

# ========== 3. 计算血管锥度 S ==========
def calculate_s_slope(lumen_folder, z_space=0.5):
    """LA = S * Distance + b"""
    files = sorted([f for f in os.listdir(lumen_folder) if f.endswith(".mhd")])
    if len(files) < 2: return 0
    
    X, Y = [], []
    for f in files:
        z = get_z_idx(f)
        la = get_2d_area_fixed(os.path.join(lumen_folder, f))
        if z is not None and la is not None:
            X.append([z * z_space])
            Y.append(la)
            
    if len(X) < 2: return 0
    model = LinearRegression().fit(X, Y)
    return model.coef_[0]

# ========== 4. 主程序 ==========
def run_paper_standard_rr(lumen_root, wall_root, min_lumen_root, output_path, z_space=0.5):
    results = []
    case_ids = [d for d in os.listdir(min_lumen_root) if os.path.isdir(os.path.join(min_lumen_root, d))]
    
    for cid in case_ids:
        # A. 病变处 (Lesion Site)：使用你提取的最小管腔切片
        lesion_dir = os.path.join(min_lumen_root, cid)
        l_files = [f for f in os.listdir(lesion_dir) if f.endswith(".mhd") and "lumen" in f]
        if not l_files: continue
        
        f_lumen_lesion = l_files[0]
        f_wall_lesion = f_lumen_lesion.replace("lumen", "wall")
        
        z_lesion = get_z_idx(f_lumen_lesion)
        la_lesion = get_2d_area_fixed(os.path.join(lesion_dir, f_lumen_lesion))
        wa_lesion = get_2d_area_fixed(os.path.join(lesion_dir, f_wall_lesion))
        owa_lesion = la_lesion + wa_lesion
        
        # B. 参考处 (Reference Site)：全段管壁最薄的地方
        wall_full_dir = os.path.join(wall_root, cid)
        ref_wall_file, wa_ref, z_ref = find_thinnest_wall_site(wall_full_dir)
        
        if ref_wall_file is None:
            print(f"跳过病例 {cid}: 无法找到参考层")
            continue
            
        # 获取对应的参考层管腔面积以计算 OWA_ref
        lumen_full_dir = os.path.join(lumen_root, cid)
        ref_lumen_file = ref_wall_file.replace("wall", "lumen")
        la_ref = get_2d_area_fixed(os.path.join(lumen_full_dir, ref_lumen_file))
        owa_ref = la_ref + wa_ref
        
        # C. 锥度修正 S
        s_slope = calculate_s_slope(lumen_full_dir, z_space)
        
        # D. 计算 RR
        # 物理距离 D = (病变索引 - 参考索引) * 0.5
        dist_d = (z_lesion - z_ref) * z_space
        # 预期面积 Expected_OWA = OWA_ref + S * D
        expected_owa = owa_ref + (s_slope * dist_d)
        
        rr = owa_lesion / expected_owa if expected_owa > 0 else None
        
        # E. 分类标准
        cat = "Intermediate"
        if rr:
            if rr > 1.05: cat = "Positive"
            elif rr < 0.95: cat = "Negative"

        results.append({
            "ID": cid,
            "RR": round(rr, 4) if rr else None,
            "Category": cat,
            "Lesion_Slice": z_lesion,
            "Ref_Slice(Thinnest_Wall)": z_ref,
            "WA_Ref": round(wa_ref, 4),
            "S_Slope": round(s_slope, 6),
            "Distance_D(mm)": round(dist_d, 2),
            "OWA_Lesion": round(owa_lesion, 4),
            "Expected_OWA": round(expected_owa, 4)
        })
        
    df = pd.DataFrame(results)
    df.to_excel(output_path, index=False)
    print(f"处理完成！参考层已选定为各病例管壁最薄层面。结果保存至: {output_path}")

# 执行
if __name__ == "__main__":
    run_paper_standard_rr(
        lumen_root="PROJECT_ROOT/lumen",
        wall_root="PROJECT_ROOT/wall",
        min_lumen_root="PROJECT_ROOT/T1_MinLumen_slice",
        output_path="PROJECT_ROOT/feature_excel/RR_Paper_Standard.xlsx",
        z_space=0.5
    )

处理完成！参考层已选定为各病例管壁最薄层面。结果保存至: PROJECT_ROOT/feature_excel/RR_Paper_Standard.xlsx
